# Clase 20 — Despliegue en AWS con EC2 y Docker Compose

**Módulo 3 — De local a producción**

**Intención:** ejecutar una aplicación de predicción en un servidor remoto y acceder a ella desde otra computadora.

## Antes de trabajar

Desde la raíz de tu copia del repositorio del curso, revisa `git status` y ejecuta `git pull` desde `main`. Después actualiza el ambiente con `uv sync --locked`. Si Git señala cambios locales que bloquean la actualización, consérvalos y consulta al profesor antes de continuar.

## 1. Punto de partida

La actividad es **individual**: cada estudiante desplegará la aplicación en su propia instancia EC2.

Utiliza la API, la interfaz y `docker-compose.yaml` de la [clase anterior](clase-19-compose.ipynb), en `labs/trabajo-local/clase-16/`. La aplicación utiliza un único `.env` junto al YAML.

Guardarás los archivos de la aplicación en **`pcd-entregas-2026/actividades/clase-20-ec2/`** y los recuperarás desde GitHub en EC2. Esta actividad practica el flujo Git para reproducir un despliegue. El `.env` real y la llave `.pem` permanecen fuera de Git.

El profesor te proporcionará las credenciales de AWS. **Cambia la contraseña inicial al ingresar por primera vez.**

Necesitas tu repositorio privado de entregas, un navegador y una terminal: **Git Bash en Windows**, Zsh en macOS o Bash en Linux.

**Resultado observable:** una predicción desde tu instancia, ejecutando la aplicación guardada en tu repositorio.

> Distingue la terminal local de la terminal SSH: la primera ejecuta comandos en tu computadora; la segunda, en EC2.

## 2. ¿Qué cambia al desplegar?

**Amazon EC2** proporciona servidores virtuales llamados **instancias**. Cada instancia tiene sistema operativo, procesador, memoria, disco y conexión de red.

Desplegar consiste en preparar ese servidor y ejecutar allí nuestra aplicación. Docker y Compose harán el mismo trabajo que en tu computadora, ahora dentro de EC2.

### ¿Qué dirección utiliza cada componente?

En tu navegador, `localhost` representa tu computadora. Para abrir Streamlit en EC2 utilizarás la **dirección IPv4 pública de la instancia**.

Dentro de EC2, API y UI conservan `network_mode: host` y comparten la red del servidor. Por eso Streamlit sigue consultando la API mediante `127.0.0.1`.

- **API:** escucha en `127.0.0.1`, sólo dentro del servidor.
- **Streamlit:** escucha en `0.0.0.0`, en las interfaces de red del servidor. El grupo de seguridad controlará quién puede entrar.

## 3. Crear la instancia

**En la consola AWS:**

1. Entra con las credenciales proporcionadas por el profesor y cambia la contraseña inicial si aún no lo has hecho.
2. En el selector de región, elige **US East (N. Virginia) — `us-east-1`**. La región es la ubicación de los recursos en AWS; conserva esta selección durante la práctica.
3. Abre **EC2 → Instances → Launch instances**.

Si ya tienes una instancia asignada, verifica su configuración sin crear otra.

### 3.1. Nombre

En **Name**, escribe tu nombre y apellido. Puedes usar el formato `nombre-apellido`.

Este nombre permite reconocer la instancia en la consola.

![Campo Name en la creación de una instancia EC2](../assets/modulo-03-produccion/clase-20/nombre-instancia.png)

Sustituye `my-instance` por tu nombre y apellido.

### 3.2. Sistema operativo

Una **AMI (Amazon Machine Image)** es la imagen inicial del sistema operativo y su configuración. Es distinta de las imágenes Docker que construiremos dentro del servidor.

En **Application and OS Images** selecciona:

- **Ubuntu Server 24.04 LTS**, de Canonical.
- Arquitectura **64-bit (x86)**.

El usuario inicial de esta imagen es `ubuntu`; lo usaremos para conectarnos.

![Opciones de sistemas operativos en Quick Start, incluida Ubuntu](../assets/modulo-03-produccion/clase-20/selector-ubuntu.jpg)

En **Quick Start**, pulsa **Ubuntu**; después verifica la versión y arquitectura indicadas arriba.

### 3.3. Tipo de instancia

El **tipo de instancia** determina sus recursos de cómputo y memoria.

En **Instance type**, selecciona **t3.medium**: 2 procesadores virtuales —vCPU— y 4 GiB de RAM, salvo que el profesor indique otro tamaño.

Esta selección deja margen para construir las imágenes y cargar el modelo. La etiqueta “Free tier” por sí sola no confirma que el uso esté cubierto.

### 3.4. Llave de acceso

Un **par de llaves** permite conectarse por SSH. La parte pública queda en la instancia; la privada se guarda en tu computadora.

En **Key pair → Create new key pair**:

1. Nombre: `nombre-apellido-key`, con tu nombre y apellido.
2. Tipo: **RSA**.
3. Formato: **`.pem`**.
4. Descarga la llave y guárdala fuera del repositorio.

Si recibiste una llave preparada, selecciona ese par en lugar de crear otro. La llave privada no se sube a Git ni se copia dentro de la aplicación.

![Opciones RSA y formato pem para crear la llave de acceso](../assets/modulo-03-produccion/clase-20/opciones-llave.jpg)

### 3.5. Red y grupo de seguridad

Una **VPC** es una red virtual de AWS; una **subred** es un segmento de esa red.

En **Network settings → Edit**:

1. Selecciona la VPC predeterminada y una de sus subredes públicas, o la red asignada para el curso.
2. En **Auto-assign public IP**, selecciona **Enable**.
3. Usa el grupo de seguridad asignado o crea `nombre-apellido-sg`.

El **grupo de seguridad** controla las conexiones permitidas mediante reglas de protocolo, puerto y origen.

**Regla para conectarte al servidor:**

- Tipo: **SSH**.
- Puerto: **22**.
- Origen: **My IP / Mi IP**.

**Regla para abrir Streamlit:**

- Tipo: **Custom TCP**.
- Puerto: **8501**.
- Origen: la IP pública de las computadoras que probarán la interfaz.

`My IP` obtiene la dirección pública de tu conexión, que puede ser compartida por varias computadoras del salón. Para autorizar una sola dirección, AWS utiliza `IP_PUBLICA/32`; `/32` limita la regla a esa IPv4.

No abras el puerto 8000: la UI consulta la API dentro de EC2. Conserva la salida de red necesaria para descargar paquetes, imágenes y el modelo.

![Encabezado Network settings con el botón Edit a la derecha](../assets/modulo-03-produccion/clase-20/configuracion-red.jpg)

El botón **Edit** abre la configuración de red y las reglas descritas arriba.

### 3.6. Disco y lanzamiento

El **volumen EBS** es el disco de la instancia. Guarda Ubuntu, archivos e imágenes Docker y se conserva cuando detienes la máquina.

1. En **Configure storage**, configura **20 GiB, gp3**, salvo otra capacidad indicada por el profesor. `gp3` es un tipo de disco SSD.
2. Comprueba que **Number of instances** sea **1**.
3. Revisa el resumen y pulsa **Launch instance**.
4. En **Instances**, espera el estado **Running** y las comprobaciones de estado aprobadas.

Si no encuentras la red o los recursos indicados, consulta al profesor antes de cambiar la configuración.

**Comprueba:** identifica tu instancia, su región y su dirección IPv4 pública. Si no tiene dirección pública, revisa la configuración antes de intentar SSH.

## 4. Conectarse por SSH

**SSH (*Secure Shell*)** abre una terminal cifrada en otra computadora. `ssh -V` muestra la versión del cliente instalado. Ejecútalo en **tu computadora**; si no existe el comando, revisa la instalación de Git Bash u OpenSSH antes de continuar.

En la consola, selecciona tu instancia y abre **Connect → SSH client**. Esa pantalla muestra el usuario y la dirección de conexión. Para la AMI de Ubuntu, el usuario inicial es **`ubuntu`**.

![Pestañas de conexión de EC2 con SSH client seleccionada](../assets/modulo-03-produccion/clase-20/ssh-client.jpg)

Selecciona **SSH client**.

### Preparar la llave

**macOS o Linux:** desde la carpeta donde guardaste la llave, ejecuta:

```bash
chmod 400 nombre-apellido-key.pem
```

`chmod` cambia permisos; `400` permite que sólo el propietario lea el archivo.

**Windows:** abre las propiedades del `.pem` en el Explorador: **Seguridad → Opciones avanzadas → Deshabilitar herencia**. Deja lectura sólo para tu usuario y retira permisos heredados de otros usuarios sobre ese archivo. Si no puedes ajustar los permisos, consulta al profesor. Git Bash utiliza la misma sintaxis de conexión, pero `chmod` no siempre modifica los permisos de Windows que revisa OpenSSH.

### Abrir la terminal remota

En los comandos, `RUTA_LLAVE` y `IP_PUBLICA` son marcadores: reemplázalos por la ubicación de tu `.pem` y la dirección de la instancia. Conserva las comillas de la ruta si contiene espacios.

*En tu computadora — Git Bash, Zsh o Bash:*

```bash
ssh -i "RUTA_LLAVE" ubuntu@IP_PUBLICA
```

`-i` indica el archivo de identidad y `usuario@dirección` señala a quién y dónde conectarse. La primera conexión muestra una **huella digital** del servidor. Comprueba que la dirección corresponde a tu instancia; puedes contrastar la huella con **Actions → Monitor and troubleshoot → Get system log**, como explica [AWS](https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/connection-prereqs-general.html). Si coincide, escribe `yes`.

*En EC2 — después de conectarte:*

```bash
whoami
pwd
cat /etc/os-release
```

`whoami` muestra el usuario; `pwd`, la ubicación actual; `cat` muestra el contenido de un archivo. Espera `ubuntu`, una carpeta como `/home/ubuntu` y Ubuntu **24.04**.

**Comprueba:** reconoce el cambio de terminal. Los comandos siguientes de instalación se ejecutan en EC2, no en tu computadora.

## 5. Instalar Docker y Compose en Ubuntu

*En EC2.* Ubuntu utiliza **APT** para instalar paquetes del sistema. `sudo` ejecuta un comando con permisos de administración. `apt update` actualiza el catálogo de paquetes; `apt install` los instala. La opción `-y` acepta la confirmación de instalación.

Esta instalación corresponde a una instancia nueva con **Ubuntu 24.04 x86_64**. Si `sudo docker --version` y `sudo docker compose version` ya funcionan, verifica sus versiones y continúa en la sección 6.

### 5.1. Preparar el repositorio de paquetes

Un repositorio APT ofrece paquetes del sistema; no es el repositorio Git de la aplicación. La llave de firma permite comprobar que los paquetes provienen de Docker.

```bash
sudo apt update
sudo apt install -y ca-certificates curl
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc
```

- `ca-certificates`: permite verificar conexiones HTTPS.
- `curl`: descarga la llave. `-f` detecta errores HTTP, `-sS` conserva errores sin mostrar progreso, `-L` sigue redirecciones y `-o` indica dónde guardar el archivo.
- `install -d`: crea la carpeta de llaves; `-m 0755` fija sus permisos de acceso.
- `chmod a+r`: permite leer la llave pública de firma.

El bloque siguiente escribe la configuración del repositorio. `tee` escribe un archivo con permisos de `sudo`; `<<'EOF'` indica que las líneas hasta `EOF` son su contenido. Copia el bloque completo, con `EOF` solo en la última línea.

```bash
sudo tee /etc/apt/sources.list.d/docker.sources <<'EOF'
Types: deb
URIs: https://download.docker.com/linux/ubuntu
Suites: noble
Components: stable
Architectures: amd64
Signed-By: /etc/apt/keyrings/docker.asc
EOF
```

`noble` es el nombre de Ubuntu 24.04 y `amd64` identifica la arquitectura x86_64 elegida al crear EC2.

### 5.2. Instalar y comprobar

*En EC2:*

```bash
sudo apt update
sudo apt install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin
sudo docker run --rm hello-world
sudo docker compose version
```

Los paquetes incluyen el motor, el cliente y los complementos para construir imágenes y ejecutar Compose. Usaremos **`docker compose`**, con espacio.

`hello-world` descarga y ejecuta un contenedor pequeño; `--rm` lo elimina al terminar. Espera el mensaje **Hello from Docker!**. El último comando debe mostrar una versión de Compose. No necesitamos instalar Docker Desktop en EC2.

**Comprueba:** Docker y Compose responden en la instancia. Si aparece un error de conexión con el motor, consulta el estado con `sudo systemctl status docker`; `systemctl` administra servicios de Ubuntu. Si está detenido, inícialo con `sudo systemctl start docker`.

Fuente del procedimiento: [instalación oficial de Docker en Ubuntu](https://docs.docker.com/engine/install/ubuntu/).

## 6. Actividad — Guardar la aplicación en tu repositorio

*En tu computadora.* Abre tu copia de **`pcd-entregas-2026`** en una segunda ventana de VS Code y abre allí una terminal. Conserva abierta la conexión SSH.

### 6.1. Preparar la carpeta

Desde la raíz del repositorio de entregas, ejecuta `git status`. Si hay cambios de otra actividad, consérvalos y resuelve su estado antes de cambiar de rama. Con el repositorio limpio:

```bash
git switch main
git pull
git switch -c feat/clase-20-ec2
mkdir -p actividades/clase-20-ec2
```

`switch -c` crea la rama de la actividad; `mkdir -p` crea la carpeta y sus directorios superiores si faltan. Usa el [flujo Git del curso](../docs/flujo-tareas.md).

Con el Explorador de archivos o Finder, **copia únicamente** `api/`, `ui/` y `docker-compose.yaml` desde `labs/trabajo-local/clase-16/` a la nueva carpeta. No copies `.env`, llaves, `.venv`, `.git`, cachés ni modelos descargados. Conserva los scripts, Dockerfiles, `.dockerignore` y `requirements.txt` existentes.

Los requisitos ya provienen del lock del curso. Python 3.12 se ejecutará dentro de las imágenes; no necesitas copiar el ambiente ni generar otro proyecto Python.

**Comprueba:** `pwd` y `git status` corresponden a `pcd-entregas-2026`; la app está en `actividades/clase-20-ec2/`.

### 6.2. Separar configuración y secretos

En `actividades/clase-20-ec2/`, crea **`.gitignore`** con:

```text
.env
.env.*
!.env.example
*.pem
.venv/
__pycache__/
api/artefactos/
```

Estas reglas excluyen configuración privada, llaves y archivos locales. `!` permite conservar `.env.example`, que sólo contiene una plantilla. Si un archivo ya estaba rastreado, `.gitignore` no lo retira: consulta al profesor si encuentras secretos en un commit.

Crea **`.env.example`** en la misma carpeta:

```text
DATABRICKS_HOST=https://TU_WORKSPACE
DATABRICKS_TOKEN=REEMPLAZAR_EN_EL_SERVIDOR
MLFLOW_TRACKING_URI=databricks
MLFLOW_REGISTRY_URI=databricks-uc
MODELO_REGISTRADO=TU_CATALOGO.TU_ESQUEMA.TU_MODELO
ALIAS_MODELO=champion
API_URL=http://127.0.0.1:8000/api/v1/predicciones
```

No pegues valores reales en esta plantilla. El `.env` privado se creará en EC2 con el acceso a tu workspace y el nombre y alias de tu modelo.

En `docker-compose.yaml`, ambos servicios conservan **`env_file: .env`**. La UI consulta la API con `127.0.0.1`; `API_URL` no lleva la IP pública de EC2. El `.env` queda fuera de los contextos `api/` y `ui/`, sin incorporarse a las imágenes.

Desde la raíz de la actividad, ejecuta:

```bash
git check-ignore .env
```

`check-ignore` comprueba qué rutas excluye Git; debe mostrar `.env`.

### 6.3. Revisar y guardar la aplicación

La carpeta de la actividad debe contener:

```text
actividades/clase-20-ec2/
├── .gitignore
├── .env.example
├── README.md
├── docker-compose.yaml
├── api/
│   ├── api.py
│   ├── Dockerfile
│   ├── requirements.txt
│   └── .dockerignore
└── ui/
    ├── streamlit_app.py
    ├── Dockerfile
    ├── requirements.txt
    └── .dockerignore
```

Conserva `network_mode: host`. En `command` de `api`, usa **`--host 127.0.0.1`**; en el de `ui`, **`--server.address=0.0.0.0`**. Con red host no necesitas `ports`.

Conserva las imágenes `nyc-taxi-api-image:v1` / `nyc-taxi-ui-image:v1` y los contenedores `nyc-taxi-api-container` / `nyc-taxi-ui-container`.

Crea un **`README.md`** breve: qué hace la app, variables que requiere, comandos para arrancar/detener y cómo comprobar una predicción. No incluyas credenciales ni direcciones personales.

Desde la **carpeta `actividades/clase-20-ec2/` de tu repositorio**, en tu computadora:

```bash
git status
git add .gitignore .env.example README.md docker-compose.yaml api ui
git diff --staged --stat
git diff --staged
```

`--staged` muestra lo preparado para el commit; `--stat` resume archivos y cambios. **Antes de continuar, comprueba que no hay `.env` reales, tokens, llaves ni archivos ajenos a la actividad.**

```bash
git commit -m "feat(clase-20): preparar aplicacion para EC2"
git push -u origin feat/clase-20-ec2
```

Abre en GitHub un **PR hacia `main`**, completa la [plantilla del curso](../docs/plantillas/pull-request-tarea.md) y revisa **Files changed**. Haz merge y confirma que aparece **Merged**. Después, en tu computadora:

```bash
git switch main
git pull
```

**Comprueba:** GitHub muestra en `main` el código y `.env.example`, sin secretos. EC2 recuperará esa versión.

## 7. Recuperar la aplicación desde GitHub en EC2

### 7.1. Acceder a tu repositorio privado

Un **token de acceso personal** permite autenticar Git por HTTPS. Para esta práctica, crea uno temporal y de sólo lectura en **tu cuenta de GitHub**, desde el navegador de tu computadora:

1. **Settings → Developer settings → Personal access tokens → Fine-grained tokens → Generate new token**.
2. Elige tu cuenta como **Resource owner** y una fecha de vencimiento corta que cubra la práctica.
3. En **Repository access**, elige **Only select repositories** y selecciona únicamente `pcd-entregas-2026`.
4. En **Repository permissions**, asigna **Contents: Read-only**. No agregues permisos de escritura.
5. Genera el token y consérvalo de forma privada para el siguiente paso. No lo incluyas en URLs, comandos, archivos ni capturas.

La contraseña de tu cuenta de GitHub no se usa para clonar por HTTPS. Consulta la [guía oficial de tokens](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens).

### 7.2. Clonar y entrar a la actividad

*En EC2 — terminal SSH.* Instala Git y el editor nano; comprueba la versión de Git:

```bash
sudo apt update
sudo apt install -y git nano
git --version
cd ~
```

**Clonar** crea una copia del repositorio. Reemplaza `TU_USUARIO` por tu usuario de GitHub y ejecuta:

```bash
git -c credential.helper= clone https://github.com/TU_USUARIO/pcd-entregas-2026.git
```

`-c credential.helper=` desactiva el guardado de credenciales mediante asistentes para este comando. Cuando Git solicite **Username**, escribe tu usuario; en **Password**, pega el token. La terminal no muestra caracteres al introducirlo.

```bash
cd ~/pcd-entregas-2026/actividades/clase-20-ec2
pwd
ls -a
git rev-parse --short HEAD
```

`rev-parse --short HEAD` muestra un identificador corto del commit descargado. Espera la carpeta de la actividad, `api/`, `ui/`, YAML y `.env.example`. El `.env` real no aparece porque no está en Git.

**Comprueba:** la actividad existe en el `main` de GitHub y en EC2. Si ya clonaste el repositorio, entra a su raíz, revisa `git status` y, si está limpio, usa `git pull` en lugar de clonarlo de nuevo. No descartes cambios locales.

### 7.3. Crear la configuración privada

*En EC2 — carpeta `clase-20-ec2`.* Copia la plantilla, limita sus permisos y ábrela en el editor de terminal:

```bash
cp .env.example .env
chmod 600 .env
nano .env
```

`cp` crea la copia; `chmod 600` permite leer y escribir sólo a su propietario. En **nano**, sustituye los marcadores por los valores de tu modelo. Guarda con **Ctrl+O**, confirma con **Enter** y sal con **Ctrl+X**. Conserva `API_URL` con `127.0.0.1`.

```bash
git check-ignore .env
git status --short
```

El primer comando debe mostrar `.env`; el segundo no debe mostrarlo como archivo nuevo. No imprimas su contenido ni compartas sus valores.

**Comprueba:** `.env` está junto al YAML y contiene los valores reales únicamente en EC2.

## 8. Construir y ejecutar en EC2

*En EC2 — `~/pcd-entregas-2026/actividades/clase-20-ec2/`.* `config --quiet` valida la configuración sin imprimir los valores del entorno. Si falla, corrige la ruta, el YAML o el `.env` antes de construir.

```bash
sudo docker compose config --quiet
sudo docker compose up --build
```

`up --build` construye las imágenes en la arquitectura del servidor y ejecuta los servicios. Puede tardar varios minutos la primera vez. La terminal queda mostrando los logs; déjala abierta.

Espera a que la API termine de cargar el modelo y muestre **Application startup complete**. `depends_on` ordena el arranque, pero no garantiza que el modelo ya esté listo.

### Comprobar la API desde el propio servidor

Abre otra conexión SSH desde una terminal local con el mismo comando de la sección 4. En esa segunda terminal **EC2**, ejecuta:

```bash
curl http://127.0.0.1:8000/health
```

`curl` solicita el recurso HTTP; espera una respuesta con `status: ok` y `version_api: v1`. Si hay un error, consulta los logs de la primera terminal antes de modificar reglas de red.

**Comprueba:** los dos contenedores arrancan y `/health` responde dentro de EC2.

## 9. Abrir Streamlit y obtener una predicción

**En el navegador de tu computadora**, reemplaza `IP_PUBLICA` por la dirección actual de la instancia:

```text
http://IP_PUBLICA:8501
```

Usa `http`: esta práctica no configura un certificado HTTPS. El grupo de seguridad debe permitir el puerto 8501 desde tu conexión.

### Probar la aplicación

1. Captura estos datos:
   - Distancia: **4.2 km**.
   - Pasajeros: **2**.
   - Hora de recolección: **18**.
   - Zona de origen: **75**.
   - Zona de destino: **42**.
2. Pulsa **Estimar duración**.
3. Comprueba que aparecen la duración y los metadatos del modelo. El número depende de la versión cargada.
4. Pide a otra persona que pruebe desde una computadora autorizada por el grupo de seguridad.

Si esa persona no puede entrar, revisa su IP pública de origen.

**Comprueba:** la aplicación obtiene una predicción desde otra computadora. Abrir la página de Streamlit sólo verifica que la interfaz está disponible.

## 10. Consultar estado y detener

### Revisar los contenedores

**En EC2, desde la segunda terminal SSH:**

Entra en la carpeta del YAML antes de consultar los contenedores:

```bash
cd ~/pcd-entregas-2026/actividades/clase-20-ec2
sudo docker compose ps
sudo docker compose logs --tail 30 api
```

`ps` muestra el estado de los servicios. `logs --tail 30 api` muestra las últimas 30 líneas de la API; cambia `api` por `ui` para revisar Streamlit.

### Detener la aplicación

1. Pulsa **Ctrl+C** en la terminal donde ejecutaste `up --build`.
2. Retira los contenedores y cierra la conexión:

```bash
sudo docker compose down
exit
```

`down` retira los contenedores del proyecto y conserva archivos e imágenes. `exit` cierra esa conexión SSH; cierra también la segunda cuando termines.

### Cerrar el acceso temporal a GitHub

En GitHub, vuelve a **Settings → Developer settings → Personal access tokens → Fine-grained tokens** y elimina el token creado para esta práctica. El código ya descargado y los contenedores siguen funcionando; para futuros `git pull` necesitarás autenticarte otra vez.

### Detener la instancia

En **CONSOLA AWS**, selecciona tu instancia y pulsa **Instance state → Stop instance**. Espera **Stopped**.

Cerrar SSH o detener contenedores no detiene la máquina. **Stop** conserva el disco EBS, que puede seguir generando costo. **Terminate** elimina la instancia y los volúmenes marcados para eliminarse; úsalo sólo cuando se indique el cierre definitivo.

**Comprueba:** la consola muestra `Stopped`. Al arrancar otra vez, consulta la IPv4 pública actual y vuelve a ejecutar Compose.

Referencia: [detener y arrancar EC2](https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/Stop_Start.html).

## 11. Si algo falla

### No puedes conectarte por SSH

- **`Connection timed out`:** revisa la dirección pública actual, el estado de EC2 y la regla SSH desde tu IP.
- **`Permission denied (publickey)`:** comprueba el usuario `ubuntu`, la llave correspondiente y los permisos del `.pem`.

### No puedes clonar o falta la actividad

- **`Repository not found` o `Authentication failed`:** revisa el usuario, repositorio, vigencia del token y acceso **Contents: Read-only** al repositorio seleccionado. No vuelvas público el repositorio.
- **La carpeta ya existe:** entra al clon existente y revisa `git status`; no lo borres para repetir el paso.
- **Falta `actividades/clase-20-ec2/`:** confirma que el PR está fusionado a `main` y que el clon está actualizado.
- **Falta `.env`:** créalo desde `.env.example` en EC2 y completa sus valores. Git no descarga secretos.

### Los contenedores no arrancan

- **Permisos de Docker:** usa `sudo` en EC2.
- **Error al cargar el modelo:** revisa logs, acceso a Databricks, nombre/alias o carpeta del modelo local.
- **`no space left on device`:** `df -h` muestra el espacio de los discos. Consulta al profesor antes de eliminar archivos o ampliar el volumen.
- **`Killed`:** puede faltar RAM. `free -h` muestra la memoria disponible; revisa el tamaño asignado con el profesor.

### La interfaz no funciona

**No abre:** comprueba `http://IP_PUBLICA:8501`, la regla de ingreso y que Streamlit escuche en `0.0.0.0`.

**Abre pero no predice:** espera a que la API cargue el modelo, revisa sus logs y comprueba `API_URL` con `127.0.0.1`.

Cambia una cosa a la vez y repite la comprobación.

## Reto opcional — Recuperar la aplicación

Después de detener la instancia, vuelve a arrancarla, consulta su dirección actual, conéctate por SSH y ejecuta Compose desde la carpeta del YAML. Obtén de nuevo una predicción.

Explica qué se conservó en el disco y qué hubo que arrancar otra vez. Apóyate en [estados de EC2](https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/ec2-instance-lifecycle.html) y [referencia de Compose up](https://docs.docker.com/reference/cli/docker/compose/up/).

## Recopilación

EC2 aporta la máquina; Ubuntu, el sistema operativo; Docker y Compose ejecutan la aplicación. GitHub conserva el código que clonamos en EC2; `.env.example` documenta la configuración y el `.env` real permanece fuera de Git. Compose la entrega al arrancar los contenedores.

El navegador usa la dirección pública de EC2. La UI consulta la API mediante `localhost` porque ambas comparten la red del servidor. Poder abrir la interfaz y poder predecir son comprobaciones distintas.

Desplegar también implica reconocer qué está ejecutándose, consultar errores y detener los recursos cuando termina su uso.

## ✅ Check final de la clase

- [ ] Distingo la terminal de mi computadora de la terminal SSH de EC2.
- [ ] Identifico AMI, tipo de instancia, disco y grupo de seguridad.
- [ ] La aplicación está en `main` de `pcd-entregas-2026`, con su PR fusionado.
- [ ] `.env.example` está en Git; `.env`, tokens y llaves permanecen fuera.
- [ ] EC2 ejecuta la aplicación clonada desde ese repositorio.
- [ ] Docker y Compose funcionan en Ubuntu.
- [ ] La API responde a `/health` dentro de EC2.
- [ ] Otra computadora autorizada obtiene una predicción en Streamlit.
- [ ] Explico por qué `API_URL` conserva `127.0.0.1`.
- [ ] Eliminé el token temporal de acceso a GitHub.
- [ ] Confirmé que mi instancia quedó `Stopped` al finalizar.

Estas comprobaciones sirven para identificar dudas; no requieren una entrega en Canvas.

## Referencias

- AWS: [tipos de instancia](https://docs.aws.amazon.com/ec2/latest/instancetypes/gp.html), [lanzar una instancia](https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/EC2_GetStarted.html), [conexión SSH](https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/connect-linux-inst-ssh.html) y [grupos de seguridad](https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/ec2-security-groups.html).
- Docker: [instalación en Ubuntu](https://docs.docker.com/engine/install/ubuntu/) y [red host](https://docs.docker.com/engine/network/drivers/host/).
- Jorge Alejandro García Martínez: [EC2 Basics — Diseño de Sistemas Escalables, ITESO](https://github.com/gamaware/system-design-course/tree/871a66a272f7c7699488498c8f9b3afca7dd049e/02-compute-ec2-basics). Lectura complementaria sobre selección de recursos, conexión y estados; utiliza Amazon Linux, cuyo usuario inicial es diferente del de Ubuntu.
- GitHub: [clonar un repositorio](https://docs.github.com/en/repositories/creating-and-managing-repositories/cloning-a-repository) y [tokens de acceso personal](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens).